In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Pull data from the database
conn = sqlite3.connect('../data/credit.db')
df = pd.read_sql('SELECT * FROM clients', conn)

print(df.shape)

(30000, 24)


In [2]:
pay_cols = ['pay_sept', 'pay_aug', 'pay_jul', 'pay_jun', 'pay_may', 'pay_apr']
bill_cols = ['bill_sept', 'bill_aug', 'bill_jul', 'bill_jun', 'bill_may', 'bill_apr']
paid_cols = ['paid_sept', 'paid_aug', 'paid_jul', 'paid_jun', 'paid_may', 'paid_apr']

# How many months were they late at all?
df['months_late'] = (df[pay_cols] > 0).sum(axis=1)

# Worst delinquency across the 6 months
df['max_delinquency'] = df[pay_cols].max(axis=1)

# Credit utilisation — how much of their limit are they using?
df['avg_utilisation'] = df[bill_cols].mean(axis=1) / df['credit_limit']

# Payment ratio — are they paying off what they owe?
df['payment_ratio'] = df[paid_cols].sum(axis=1) / (df[bill_cols].sum(axis=1) + 1)

print(df[['months_late', 'max_delinquency', 'avg_utilisation', 'payment_ratio']].describe())

        months_late  max_delinquency  avg_utilisation  payment_ratio
count  30000.000000     30000.000000     30000.000000   30000.000000
mean       0.834200         0.438733         0.373048      21.039708
std        1.554303         1.345154         0.351890    1255.771376
min        0.000000        -2.000000        -0.232590    -589.000000
25%        0.000000         0.000000         0.029997       0.041073
50%        0.000000         0.000000         0.284834       0.086192
75%        1.000000         2.000000         0.687929       0.598218
max        6.000000         8.000000         5.364308  162000.000000


In [3]:
# Cap payment_ratio at a sensible range
df['payment_ratio'] = df['payment_ratio'].clip(lower=0, upper=2)

# Cap utilisation at a sensible range
df['avg_utilisation'] = df['avg_utilisation'].clip(lower=0, upper=2)

print(df[['avg_utilisation', 'payment_ratio']].describe())

       avg_utilisation  payment_ratio
count     30000.000000   30000.000000
mean          0.372851       0.349944
std           0.349678       0.458101
min           0.000000       0.000000
25%           0.029997       0.041073
50%           0.284834       0.086192
75%           0.687929       0.598218
max           2.000000       2.000000


In [4]:
# Split features and target
X = df.drop('defaulted', axis=1)
y = df['defaulted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train default rate: {y_train.mean():.3f}")
print(f"Test default rate: {y_test.mean():.3f}")

Train: 24000, Test: 6000
Train default rate: 0.221
Test default rate: 0.221


In [5]:
# Baseline
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                            class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# XGBoost
scale = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                    scale_pos_weight=scale, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)

# Compare
for name, model in [('Logistic Regression', lr), ('Random Forest', rf), ('XGBoost', xgb)]:
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    print(f"\n=== {name} ===")
    print(f"ROC-AUC: {roc_auc_score(y_test, proba):.3f}")
    print(classification_report(y_test, pred, target_names=['No Default', 'Default']))

C:\Users\Abdul\PycharmProjects\PythonProject5\credit-default-risk-analytics\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



=== Logistic Regression ===
ROC-AUC: 0.738
              precision    recall  f1-score   support

  No Default       0.87      0.78      0.83      4673
     Default       0.44      0.60      0.51      1327

    accuracy                           0.74      6000
   macro avg       0.66      0.69      0.67      6000
weighted avg       0.78      0.74      0.76      6000


=== Random Forest ===
ROC-AUC: 0.775
              precision    recall  f1-score   support

  No Default       0.88      0.82      0.85      4673
     Default       0.49      0.61      0.54      1327

    accuracy                           0.77      6000
   macro avg       0.68      0.71      0.69      6000
weighted avg       0.79      0.77      0.78      6000


=== XGBoost ===
ROC-AUC: 0.777
              precision    recall  f1-score   support

  No Default       0.88      0.80      0.84      4673
     Default       0.47      0.62      0.53      1327

    accuracy                           0.76      6000
   macro avg  

In [6]:
import pickle

with open('../models/xgb_credit_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

with open('../models/rf_credit_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

print("Models saved")


Models saved
